In [ ]:
%cd /kaggle/working
!rm -rf DGM-HMamba verified data

# ۱) کد
!cp -r /kaggle/input/models/zarrinmahdavi/dgm-hmamba-main/pytorch/default/1/DGM-HMamba-main DGM-HMamba
%cd DGM-HMamba
!ls requirements.txt scripts/demo_real.py class_embeddings/glove.6B.300d.json

# ۲) داده‌ها — ⬅️ اینجا تغییر کرد: با %env تعریف می‌شود
%env DS=/kaggle/input/datasets/zarrinmahdavi/dataset
!mkdir -p data/IndianPines data/PaviaU data/Salinas
!cp $DS/Indian_pines_corrected.mat $DS/Indian_pines_gt.mat data/IndianPines/
!cp $DS/PaviaU.mat $DS/PaviaU_gt.mat data/PaviaU/
!cp $DS/salinas_corrected.mat $DS/salinas_gt.mat data/Salinas/

# ۳) مدل‌های آموزش‌دیده
!cp -r /kaggle/input/datasets/zarrinmahdavi/verified-ip2pu-v verified
!ls verified/s5_m42/model_final.pt

# ۴) چک‌ها
!python -c "import torch; print('GPU:', torch.cuda.is_available())"
!python scripts/demo_forward.py
print("=== SETUP COMPLETE ===")

In [ ]:
runner = r'''
#!/bin/bash
cd /kaggle/working/DGM-HMamba
mkdir -p runs /kaggle/working/backup_zips
run_exp() {
  name=$1; shift
  if [ -f "runs/$name/done.flag" ]; then echo "SKIP $name"; return; fi
  mkdir -p runs/$name                                    # ← این خط اضافه شد
  echo "=== RUNNING: $name ==="
  python scripts/demo_real.py --data-root data "$@" --output-dir runs/$name > runs/$name/log.txt 2>&1
  if [ $? -eq 0 ]; then
    touch runs/$name/done.flag; echo "=== DONE: $name ==="
    zip -qr /kaggle/working/backup_zips/$name.zip runs/$name
    echo "=== BACKUP SAVED: $name.zip ==="
  else
    echo "=== FAILED: $name ==="
  fi
}
'''
with open("/kaggle/working/runner.sh","w") as f: f.write(runner)

exp = r'''
source /kaggle/working/runner.sh
run_exp s5_m42_eval --weights verified/s5_m42/model_final.pt
run_exp quick_train --stage1-iters 300 --stage2-steps 200
run_exp verified_s5_m42 --preset verified --split-seed 5 --seed 42
run_exp verified_glove_s5_m42 --preset verified --split-seed 5 --seed 42 --class-embeddings class_embeddings/glove.6B.300d.json
run_exp support_free_s5_m42 --preset verified --split-seed 5 --seed 42 --stage2-steps 0 --class-embeddings class_embeddings/glove.6B.300d.json --support-free
run_exp anchor_feature_s5_m42 --preset verified --split-seed 5 --seed 42 --class-embeddings class_embeddings/glove.6B.300d.json --anchor-init feature
run_exp anchor_none_s5_m42 --preset verified --split-seed 5 --seed 42 --anchor-init none
echo "ALL FINISHED"
'''
with open("/kaggle/working/session.sh","w") as f: f.write(exp)

import subprocess
proc = subprocess.Popen(
    ["bash", "/kaggle/working/session.sh"],
    stdout=open("/kaggle/working/session.log", "w"),
    stderr=subprocess.STDOUT,
    cwd="/kaggle/working/DGM-HMamba"
)
print("Started! PID =", proc.pid)

In [ ]:
import subprocess
proc = subprocess.Popen(
    ["bash", "/kaggle/working/session.sh"],
    stdout=open("/kaggle/working/session.log", "w"),
    stderr=subprocess.STDOUT,
    cwd="/kaggle/working/DGM-HMamba"
)
print("Started! PID =", proc.pid)
print("First results (s5_m42_eval) in ~5 minutes.")

In [ ]:
!grep -E "RUNNING|DONE|FAILED|SKIP" /kaggle/working/session.log
!ls -lh /kaggle/working/backup_zips/ 2>/dev/null

In [ ]:
!cat /kaggle/working/session.log



In [ ]:
!cat /kaggle/working/DGM-HMamba/runs/s5_m42_eval/demo_summary.json 2>/dev/null || unzip -p /kaggle/working/backup_zips/s5_m42_eval.zip runs/s5_m42_eval/demo_summary.json

!grep -E "RUNNING|DONE|FAILED|SKIP" /kaggle/working/session.log
!ls -lh /kaggle/working/backup_zips/
!tail -3 /kaggle/working/DGM-HMamba/runs/quick_train/log.txt 2>/dev/null

import subprocess
proc = subprocess.Popen(
    ["bash", "/kaggle/working/session.sh"],
    stdout=open("/kaggle/working/session.log", "a"),
    stderr=subprocess.STDOUT,
    cwd="/kaggle/working/DGM-HMamba"
)
print("Started! PID =", proc.pid)

!grep -E "RUNNING|DONE|FAILED|SKIP" /kaggle/working/session.log
!ls -lh /kaggle/working/backup_zips/

In [ ]:
!tail -5 /kaggle/working/DGM-HMamba/runs/quick_train/log.txt 2>/dev/null

In [ ]:
!pkill -f session.sh; pkill -f demo_real.py; sleep 3
!nvidia-smi          # باید memory usage را پایین (~0) نشان دهد

In [ ]:
!kill -9 614 630 2>/dev/null; sleep 2; nvidia-smi

In [ ]:
runner = r'''
#!/bin/bash
cd /kaggle/working/DGM-HMamba
mkdir -p runs /kaggle/working/backup_zips
export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
run_exp() {
  name=$1; shift
  if [ -f "runs/$name/done.flag" ]; then echo "SKIP $name"; return; fi
  rm -rf runs/$name
  mkdir -p runs/$name
  echo "=== RUNNING: $name ==="
  python scripts/demo_real.py --data-root data --grad-checkpoint --map-batch 64 "$@" --output-dir runs/$name > runs/$name/log.txt 2>&1
  if [ $? -eq 0 ]; then
    touch runs/$name/done.flag; echo "=== DONE: $name ==="
    zip -qr /kaggle/working/backup_zips/$name.zip runs/$name
    echo "=== BACKUP SAVED: $name.zip ==="
  else
    echo "=== FAILED: $name ==="
  fi
}
'''
with open("/kaggle/working/runner.sh","w") as f: f.write(runner)
print("runner.sh fixed: --grad-checkpoint --map-batch 64")

In [ ]:
import subprocess
proc = subprocess.Popen(
    ["bash", "/kaggle/working/session.sh"],
    stdout=open("/kaggle/working/session.log", "a"),
    stderr=subprocess.STDOUT,
    cwd="/kaggle/working/DGM-HMamba"
)
print("Restarted! PID =", proc.pid)

In [ ]:
import subprocess
proc = subprocess.Popen(
    ["bash", "/kaggle/working/session.sh"],
    stdout=open("/kaggle/working/session.log", "a"),
    stderr=subprocess.STDOUT,
    cwd="/kaggle/working/DGM-HMamba"
)
print("Restarted! PID =", proc.pid)

In [ ]:
!grep -E "RUNNING|DONE|FAILED|SKIP" /kaggle/working/session.log | tail -8
!tail -5 /kaggle/working/DGM-HMamba/runs/quick_train/log.txt 2>/dev/null
!nvidia-smi --query-gpu=memory.used --format=csv

In [ ]:
!grep -E "DONE|FAILED" /kaggle/working/session.log | tail -5
!ls -lh /kaggle/working/backup_zips/
!tail -2 /kaggle/working/DGM-HMamba/runs/$(grep RUNNING /kaggle/working/session.log | tail -1 | awk '{print $3}')/log.txt 2>/dev/null

In [ ]:
import subprocess
proc = subprocess.Popen(
    ["bash", "/kaggle/working/session.sh"],
    stdout=open("/kaggle/working/session.log", "a"),
    stderr=subprocess.STDOUT,
    cwd="/kaggle/working/DGM-HMamba"
)
print("Restarted! PID =", proc.pid)

In [ ]:
!grep -E "DONE|FAILED" /kaggle/working/session.log | tail -6
!grep "RUNNING" /kaggle/working/session.log | tail -1
!ls -lh /kaggle/working/backup_zips/

In [ ]:
import subprocess
# قدیمی‌ترین bash session.sh را پیدا و می‌کشیم (python را نمی‌کشیم)
out = subprocess.run(["bash","-c","ps -eo pid,etimes,cmd | grep 'bash /kaggle/working/session.sh' | grep -v grep | sort -k2 -n | head -1 | awk '{print $1}'"], capture_output=True, text=True).stdout.strip()
if out:
    subprocess.run(["kill", out]); print("killed duplicate bash PID:", out)
else:
    print("no duplicate")

In [ ]:
print(1+1)

In [ ]:
import subprocess
print(subprocess.run(["bash","-c","grep -E 'DONE|FAILED' /kaggle/working/session.log | tail -8; echo ---; ls /kaggle/working/backup_zips/"], capture_output=True, text=True).stdout)

In [ ]:
import subprocess
proc = subprocess.Popen(
    ["bash", "/kaggle/working/session.sh"],
    stdout=open("/kaggle/working/session.log", "a"),
    stderr=subprocess.STDOUT,
    cwd="/kaggle/working/DGM-HMamba"
)
print("Resumed! PID =", proc.pid)

!grep -E "DONE|FAILED" /kaggle/working/session.log | tail -6
!grep "RUNNING" /kaggle/working/session.log | tail -1
!ls -lh /kaggle/working/backup_zips/

In [ ]:
!grep -E "DONE|FAILED" /kaggle/working/session.log | tail -6
!grep "RUNNING" /kaggle/working/session.log | tail -1
!ls -lh /kaggle/working/backup_zips/

In [ ]:
import json
s = json.load(open('/kaggle/working/DGM-HMamba/runs/verified_s5_m42/demo_summary.json'))
e = s['episodic_metrics']
print("5-shot OA: %.2f  AA: %.2f" % (e['5way_5shot']['oa_mean']*100, e['5way_5shot']['aa_mean']*100))
print("1-shot OA: %.2f  AA: %.2f" % (e['5way_1shot']['oa_mean']*100, e['5way_1shot']['aa_mean']*100))
print("Map    OA: %.2f  AA: %.2f" % (s['map_metrics']['oa_mean']*100, s['map_metrics']['aa_mean']*100))

In [ ]:
!grep -E "DONE|FAILED|RUNNING" /kaggle/working/session.log | tail -4
!ls -lh /kaggle/working/backup_zips/
!nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader
!tail -2 /kaggle/working/DGM-HMamba/runs/verified_glove_s5_m42/log.txt 2>/dev/null

In [ ]:
import json
s = json.load(open('/kaggle/working/DGM-HMamba/runs/verified_s5_m42/demo_summary.json'))
e = s['episodic_metrics']
print("5-shot OA: %.2f  AA: %.2f  k: %.3f" % (e['5way_5shot']['oa_mean']*100, e['5way_5shot']['aa']*100, e['5way_5shot']['kappa']))
print("1-shot OA: %.2f  AA: %.2f  k: %.3f" % (e['5way_1shot']['oa_mean']*100, e['5way_1shot']['aa']*100, e['5way_1shot']['kappa']))
print("Map    OA: %.2f  AA: %.2f  k: %.3f" % (s['map_metrics']['oa_mean']*100, s['map_metrics']['aa_mean']*100, s['map_metrics']['kappa_mean']))

In [ ]:
!grep -E "DONE|FAILED|RUNNING" /kaggle/working/session.log | tail -4
!ls -lh /kaggle/working/backup_zips/

In [ ]:
!grep -E "DONE|FAILED|RUNNING|ALL FINISHED" /kaggle/working/session.log | tail -6
!ls /kaggle/working/DGM-HMamba/runs/*/done.flag 2>/dev/null | wc -l
!ls -lh /kaggle/working/backup_zips/

In [ ]:
import json
s = json.load(open('/kaggle/working/DGM-HMamba/runs/support_free_s5_m42/demo_summary.json'))
m = s.get('support_free') or {}
print('support-free enabled:', s['args'].get('support_free'))
print('Map OA: %.2f  AA: %.2f' % (s['map_metrics']['oa_mean']*100, s['map_metrics']['aa_mean']*100))
e = s['episodic_metrics']
print('5-shot OA: %.2f | 1-shot OA: %.2f' % (e['5way_5shot']['oa_mean']*100, e['5way_1shot']['oa_mean']*100))

In [ ]:
!grep -E "DONE|FAILED|RUNNING|ALL FINISHED" /kaggle/working/session.log | tail -4
!ls /kaggle/working/DGM-HMamba/runs/*/done.flag 2>/dev/null | wc -l
!ls -lh /kaggle/working/backup_zips/

In [ ]:
!grep -E "DONE|FAILED|RUNNING|ALL FINISHED" /kaggle/working/session.log | tail -3
!ls /kaggle/working/DGM-HMamba/runs/*/done.flag 2>/dev/null | wc -l
!ls -lh /kaggle/working/backup_zips/

In [ ]:
!grep -E "RUNNING|DONE|FAILED" /kaggle/working/session.log | tail -3
!nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader
!ls -la /kaggle/working/DGM-HMamba/runs/anchor_none_s5_m42/log.txt
!tail -2 /kaggle/working/DGM-HMamba/runs/anchor_none_s5_m42/log.txt 2>/dev/null
!date

In [ ]:
!grep "ALL FINISHED" /kaggle/working/session.log
!ls /kaggle/working/DGM-HMamba/runs/*/done.flag 2>/dev/null | wc -l
!ls -lh /kaggle/working/backup_zips/

In [ ]:
import json, glob, pandas as pd
rows = []
for p in sorted(glob.glob('/kaggle/working/DGM-HMamba/runs/*/demo_summary.json')):
    s = json.load(open(p))
    e = s['episodic_metrics']
    rows.append({'run': p.split('/')[-2],
                 'OA_5shot': round(e['5way_5shot']['oa_mean']*100, 2),
                 'OA_1shot': round(e['5way_1shot']['oa_mean']*100, 2),
                 'Map_OA': round(s['map_metrics']['oa_mean']*100, 2),
                 'Map_AA': round(s['map_metrics']['aa_mean']*100, 2)})
df = pd.DataFrame(rows).set_index('run')
print(df.to_string())
df.to_csv('/kaggle/working/comparison_table.csv')

In [ ]:
# همه خروجی‌ها را یک‌جا زیپ کن + جدول
!cd /kaggle/working/DGM-HMamba && zip -qr /kaggle/working/FINAL_all_runs.zip runs

In [ ]:
!ls -lh /kaggle/working/FINAL_all_runs.zip
!ls -lh /kaggle/working/

In [ ]:
%cd /kaggle/working
!rm -rf DGM-HMamba data
!cp -r /kaggle/input/models/zarrinmahdavi/dgm-hmamba-main/pytorch/default/1/DGM-HMamba-main DGM-HMamba
%cd DGM-HMamba
!ls requirements.txt scripts/demo_real.py class_embeddings/glove.6B.300d.json
%env DS=/kaggle/input/datasets/zarrinmahdavi/dataset
!mkdir -p data/IndianPines data/PaviaU data/Salinas
!cp $DS/Indian_pines_corrected.mat $DS/Indian_pines_gt.mat data/IndianPines/
!cp $DS/PaviaU.mat $DS/PaviaU_gt.mat data/PaviaU/
!cp $DS/salinas_corrected.mat data/Salinas/Salinas_corrected.mat
!cp $DS/salinas_gt.mat data/Salinas/Salinas_gt.mat
!ls data/Salinas/
!python -c "import torch; print('GPU:', torch.cuda.is_available())"
!python scripts/demo_forward.py
print("Setup done!")

In [ ]:
runner = r'''
#!/bin/bash
cd /kaggle/working/DGM-HMamba
mkdir -p runs /kaggle/working/backup_zips
export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
run_exp() {
  name=$1; shift
  if [ -f "runs/$name/done.flag" ]; then echo "SKIP $name"; return; fi
  rm -rf runs/$name; mkdir -p runs/$name
  echo "=== RUNNING: $name ==="
  python scripts/demo_real.py --data-root data --grad-checkpoint --map-batch 64 "$@" --output-dir runs/$name > runs/$name/log.txt 2>&1
  if [ $? -eq 0 ]; then
    touch runs/$name/done.flag; echo "=== DONE: $name ==="
    zip -qr /kaggle/working/backup_zips/$name.zip runs/$name
    echo "=== BACKUP SAVED ==="
  else
    echo "=== FAILED: $name ==="
  fi
}
'''
with open("/kaggle/working/runner.sh","w") as f: f.write(runner)

exp = r'''
source /kaggle/working/runner.sh
# Task 2: Salinas -> PaviaU  (ترتیب: ضروری ها اول)
run_exp s2p_verified     --source salinas --target pavia_u --preset verified --split-seed 5 --seed 42
run_exp s2p_glove        --source salinas --target pavia_u --preset verified --split-seed 5 --seed 42 --class-embeddings class_embeddings/glove.6B.300d.json
run_exp s2p_anchor_none  --source salinas --target pavia_u --preset verified --split-seed 5 --seed 42 --anchor-init none
run_exp s2p_anchor_feat  --source salinas --target pavia_u --preset verified --split-seed 5 --seed 42 --class-embeddings class_embeddings/glove.6B.300d.json --anchor-init feature
run_exp s2p_support_free --source salinas --target pavia_u --preset verified --split-seed 5 --seed 42 --stage2-steps 0 --class-embeddings class_embeddings/glove.6B.300d.json --support-free
echo "ALL FINISHED"
'''
with open("/kaggle/working/session.sh","w") as f: f.write(exp)
print("Scripts ready!")

In [ ]:
import subprocess
proc = subprocess.Popen(
    ["bash", "/kaggle/working/session.sh"],
    stdout=open("/kaggle/working/session.log", "w"),
    stderr=subprocess.STDOUT,
    cwd="/kaggle/working/DGM-HMamba"
)
print("Started! PID =", proc.pid)

In [ ]:
!grep -E "RUNNING|DONE|FAILED" /kaggle/working/session.log | tail -3
!tail -2 /kaggle/working/DGM-HMamba/runs/s2p_verified/log.txt 2>/dev/null
!nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader